# Uniform-sphere instability benchmark

This notebook reproduces the physics and acceptance criterion of the TOMMOS uniform-sphere regression test in an inspectable workflow. A uniformly magnetised, magnetically soft sphere is swept through the expected instability field, and the numerical transition is compared with the benchmark value $B_\mathrm{inst}=J_s/3$.

The current workflow deliberately exposes the TOMMOS preprocessing, private adapter input writer, direct runner call, and manual result import. Comments mark the places that should later be replaced by the public `tommos-adapter` workflow. General body-mesh generation remains an explicit TOMMOS preprocessing step; it is not naturally part of `HysteresisDriver.drive`.

## Physical benchmark

The regression configuration uses a sphere with diameter $20\,\mathrm{nm}$, saturation polarisation $J_s=\mu_0 M_s=1\,\mathrm{T}$, exchange constant $A=0$, and anisotropy constant $K_1=0$. For the uniform mode of an ideal sphere, the demagnetising factor along every principal direction is $1/3$, which motivates the reference field

$$B_\mathrm{inst}=\frac{J_s}{3}=0.333\ldots\,\mathrm{T}.$$

The finite tetrahedral mesh, finite air domain, and numerical minimiser shift the observed transition. The regression therefore uses a 10% relative tolerance. It detects the loss of the uniform state; it does **not** establish that the emerging state has vortex topology. If $A$ or $K_1$ is changed below, the notebook switches to an exploratory evaluation and no longer applies $J_s/3$ as an analytical reference.

In [ ]:
import contextlib
import os
import shlex
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

# Avoid the macOS OpenBLAS/JAX threading failure observed in TOMMOS CPU runs.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
# Trame provides interactive PyVista views inside Jupyter.
pv.set_jupyter_backend("trame")

In [ ]:
name = "uniform_sphere"
diameter_nm = 20.0
mesh_size_nm = 3.0
spontaneous_polarisation = 1.0  # Js = mu0 Ms in tesla
exchange_constant = 1e-13  # J/m
anisotropy_constant = 0.001  # J/m3
field_start = 0.34
field_stop = 0.00
field_step = -0.01
uniformity_threshold = 0.99
relative_tolerance = 0.10
reference_instability_field = (
    spontaneous_polarisation / 3
    if exchange_constant == 0 and anisotropy_constant == 0
    else None
)

# A temporary directory avoids adding generated NPZ, VTU, KRN, and P2 files to Git.
temporary_directory = tempfile.TemporaryDirectory(prefix="tommos-sphere-")
run_directory = Path(temporary_directory.name)
run_directory

## Generate the magnetic-body mesh

The regression uses TOMMOS's `ellipsoid` geometry with equal extents and the MeshPy backend. The generated coordinates are in nanometres. This explicit meshing step is expected to remain separate from the hysteresis driver, although a future convenience API could wrap it.

In [ ]:
# NBVAL_SKIP
mesh_command = [
    "tommos",
    "mesh",
    "--geom",
    "ellipsoid",
    "--extent",
    f"{diameter_nm:g},{diameter_nm:g},{diameter_nm:g}",
    "--h",
    f"{mesh_size_nm:g}",
    "--backend",
    "meshpy",
    "--out-name",
    name,
]
print(shlex.join(mesh_command))
mesh_result = subprocess.run(
    mesh_command,
    cwd=run_directory,
    check=True,
    capture_output=True,
    text=True,
)
print(mesh_result.stdout)

In [ ]:
# NBVAL_SKIP
state = pv.read(run_directory / f"{name}.vtu")
state.points *= ta.scripts.DEFAULT_MESH_UNIT  # nm -> m
state.cell_data["region_id"] = state.cell_data.pop("mat_id")
state.point_data["m"] = np.tile((0.0, 0.0, 1.0), (state.n_points, 1))
state.cell_data["Ms"] = np.full(
    state.n_cells,
    spontaneous_polarisation / mm.consts.mu0,
)
print(f"Body mesh: {state.n_points} points, {state.n_cells} tetrahedra")

In [ ]:
# NBVAL_SKIP
body_nm = state.copy(deep=True)
body_nm.points /= ta.scripts.DEFAULT_MESH_UNIT
body_nm.plot(
    color="tab:orange",
    show_edges=True,
    jupyter_backend="trame",
    cpos="xz",
)

## Define the Ubermag system

Exchange and anisotropy are retained explicitly because they provide the material columns expected by the current TOMMOS translation. Their values are configured above. Only $A=K_1=0$ activates the analytical regression reference; other values are treated as an exploratory calculation. The initial magnetisation is uniform along $+z$.

In [ ]:
# NBVAL_SKIP
system = mm.System(
    name=name,
    energy=(
        mm.Exchange(A=exchange_constant)
        + mm.UniaxialAnisotropy(K=anisotropy_constant, u=(0, 0, 1))
        + mm.Demag()
    ),
    m=state,
)
system

## Write TOMMOS input and run the field sweep

The next two cells reproduce the private input-writing and runner calls used by the regression test. TOMMOS adds its default box-shaped air domain; its current default `KL=10` produces an outer extent of about $228\,\mathrm{nm}$ for this mesh. The automated test disables snapshots, whereas this notebook saves every step so that the mesh and magnetisation can be inspected.

Once automatic shell handling, field-direction semantics, and result import are available through the public adapter API, these cells should be replaced by: 

```python
driver.drive(system, Hmin=H_start, Hmax=H_stop, n=5)
```

In [ ]:
# NBVAL_SKIP
driver = ta.HysteresisDriver(max_iter=1000)
with contextlib.chdir(run_directory):
    driver._write_input_files(
        system,
        hx=0,
        hy=0,
        hz=1,
        hstart=field_start,
        hfinal=field_stop,
        hstep=field_step,
    )

for suffix in (".npz", ".krn", ".p2"):
    path = run_directory / f"{name}{suffix}"
    print(path.name, path.stat().st_size, "bytes")

In [ ]:
# NBVAL_SKIP
runner_arguments = f"{name} --add-shell --hmax 20 --snapshot-every 1"
print("tommos loop", runner_arguments)
with contextlib.chdir(run_directory):
    result = ta.TommosRunner()._call(runner_arguments)

stdout = result.stdout.decode(errors="replace")
stderr = result.stderr.decode(errors="replace")
if result.returncode != 0:
    raise RuntimeError(stdout + stderr)
print("\n".join(stdout.splitlines()[-12:]))

## Detect the loss of the uniform state

TOMMOS writes a two-column `.mh` compatibility file containing the applied field coordinate and the volume-averaged polarisation parallel to the sweep direction. The state is classified as uniform while $J_\parallel$ remains above the configurable fraction of $J_s$. If the sweep contains a stable-to-unstable crossing, the numerical instability field is the midpoint between the two bracketing field values. Otherwise, the notebook reports that no transition was found instead of assuming one exists.

Later, manual loading of this file should be replaced by the table attached to the driven Ubermag system, for example `system.table`.

In [ ]:
# NBVAL_SKIP
history_file = run_directory / f"hyst_{name}" / f"{name}.mh"
history = np.atleast_2d(np.loadtxt(history_file))
field = history[:, 0]
parallel_polarisation = history[:, 1]
stable = parallel_polarisation >= (
    uniformity_threshold * spontaneous_polarisation
)
crossings = np.flatnonzero(stable[:-1] & ~stable[1:])
numerical_instability_field = None

if not stable[0]:
    print("The initial state is already below the uniformity criterion.")
elif crossings.size:
    crossing = crossings[0]
    numerical_instability_field = np.mean(field[crossing : crossing + 2])
    print(f"Numerical instability field: {numerical_instability_field:.4f} T")
else:
    print(
        "No loss of the uniform state was detected between "
        f"{field[0]:.4f} T and {field[-1]:.4f} T."
    )

if reference_instability_field is not None:
    if numerical_instability_field is None:
        raise AssertionError("The regression sweep did not contain an instability.")
    relative_error = (
        abs(numerical_instability_field - reference_instability_field)
        / reference_instability_field
    )
    np.testing.assert_allclose(
        numerical_instability_field,
        reference_instability_field,
        rtol=relative_tolerance,
    )
    print(f"Reference instability field: {reference_instability_field:.4f} T")
    print(f"Relative error:              {relative_error:.1%}")
else:
    print("No analytical reference is applied for these material parameters.")

In [ ]:
# NBVAL_SKIP
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(field, parallel_polarisation, marker="o", label=r"$J_\parallel$")
ax.axhline(
    uniformity_threshold * spontaneous_polarisation,
    color="tab:gray",
    linestyle=":",
    label=f"{uniformity_threshold:.0%} $J_s$ criterion",
)
if reference_instability_field is not None:
    ax.axvline(
        reference_instability_field,
        color="black",
        linestyle="--",
        label=r"$J_s/3$",
    )
    ax.axvspan(
        reference_instability_field * (1 - relative_tolerance),
        reference_instability_field * (1 + relative_tolerance),
        color="tab:green",
        alpha=0.15,
        label=f"{relative_tolerance:.0%} regression tolerance",
    )
if numerical_instability_field is not None:
    ax.axvline(
        numerical_instability_field,
        color="tab:red",
        linestyle=":",
        label="numerical transition",
    )
ax.set(xlabel=r"Applied field coordinate $B$ (T)", ylabel=r"$J_\parallel$ (T)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

## Inspect the generated air domain and final state

The TOMMOS output mesh contains both the magnetic sphere and the automatically generated air domain. The largest material identifier belongs to air. For the current default box shell, the transparent outer surface below is cubic. Adding `--shell-type triangles` would instead create a faceted sphere-like shell for this spherical body, but that would define a different regression baseline and should first be studied separately.

In [ ]:
# NBVAL_SKIP
snapshot_files = sorted((run_directory / f"hyst_{name}").glob("state_cfg*.vtu"))
final_state = pv.read(snapshot_files[-1])
material_key = next(key for key in ("mat_id", "region_id") if key in final_state.cell_data)
air_id = int(np.max(final_state.cell_data[material_key]))
body_cell_ids = np.flatnonzero(final_state.cell_data[material_key] != air_id)
airbox_cell_ids = np.flatnonzero(final_state.cell_data[material_key] == air_id)
magnetic_body = final_state.extract_cells(body_cell_ids)
airbox = final_state.extract_cells(airbox_cell_ids)

extent = np.ptp(np.asarray(final_state.bounds).reshape(3, 2), axis=1)
print(f"Air material ID: {air_id}")
print(f"Full mesh extent: {extent.round(1)} nm")
print(f"Final body cells: {magnetic_body.n_cells}")

In [ ]:
# NBVAL_SKIP
result_plotter = pv.Plotter()
result_plotter.add_mesh(
    magnetic_body,
    color="tab:orange",
    show_edges=True,
)
result_plotter.add_mesh(
    airbox,
    color="lightgray",
    opacity=0.05,
)
result_plotter.add_axes()
result_plotter.show(jupyter_backend="trame", cpos="xz")

### Plot the magnetisation vector field without the Airbox

As in the multigrain tutorial, this view retains only the magnetic cells. The Airbox is therefore absent. PyVista's `glyph` filter turns the nodal `m` vectors into arrows, `tolerance` reduces their density, and the colour represents the $z$ component $m_z$.

In [ ]:
# NBVAL_SKIP
if "m" not in magnetic_body.point_data:
    raise RuntimeError("The TOMMOS snapshot does not contain point data 'm'.")

magnetic_body.point_data["m_z"] = np.asarray(
    magnetic_body.point_data["m"]
)[:, 2]
magnetisation_glyphs = magnetic_body.glyph(
    orient="m",
    scale=False,
    factor=1.5,
    tolerance=0.04,
)

vector_plotter = pv.Plotter()
vector_plotter.add_mesh(
    magnetic_body.extract_surface(algorithm="dataset_surface"),
    color="tab:orange",
    opacity=0.15,
)
vector_plotter.add_mesh(
    magnetisation_glyphs,
    scalars="m_z",
    cmap="coolwarm",
    clim=(-1, 1),
    scalar_bar_args={"title": "m_z"},
)
vector_plotter.add_axes()
vector_plotter.show(jupyter_backend="trame", cpos="xz")

## Interpretation and next steps

This calculation is useful as a regression benchmark because it is small, has a simple reference scale, and is sensitive to changes in meshing, shell generation, and minimisation. It is not a high-precision validation of an infinite-domain analytical problem. A stronger validation study would vary the body mesh size, air-domain size, shell shape, and minimiser settings and demonstrate convergence.

The intended public adapter workflow should eventually replace the private input writer, direct runner invocation, manual `.mh` import, and manual VTU handling. The explicit TOMMOS body-mesh generation can remain as a transparent preprocessing step.